# Train Your Own Neural Network: Handwritten Digits

**Goal:** teach a small neural network to read *your* handwriting, then find out what makes it better or worse.

You will work in 5 steps. Run each cell with the play button (or Shift+Enter), **in order, top to bottom**. If you see a `NameError`, you skipped a cell above: scroll up and run it first. If the notebook restarts, start again from Step 0.

1. Upload 10 of your own digit pictures (one for each digit 0-9)
2. Choose the network settings
3. Train the network
4. Test it on a brand new picture
5. Tune the settings and try again

## Step 0: Setup (run once)
This loads the network code (from Michael Nielsen's book *Neural Networks and Deep Learning*, Python 3 version) and the MNIST data set of 70,000 handwritten digits. It takes a few seconds.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, re, glob, random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
import mnist_loader, network

mnist_train, _, mnist_test = mnist_loader.load_data_wrapper()
mnist_train = list(mnist_train)
mnist_test = list(mnist_test)
print("MNIST training pictures:", len(mnist_train), "| MNIST test pictures:", len(mnist_test))

RESULTS = []   # the table of your experiments is collected here


def prepare_image(path):
    """Turn any picture of ONE digit into 784 numbers like MNIST (28x28, white digit on black)."""
    arr = np.array(Image.open(path).convert("L"), dtype=float)
    if arr.mean() > 127:                      # light background -> make it dark
        arr = 255 - arr
    ys, xs = np.where(arr > 50)               # cut away empty space around the digit
    arr = arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = arr.shape
    k = max(3, (max(h, w) // 10) // 2 * 2 + 1)   # make thin lines thicker
    img = Image.fromarray(arr.astype("uint8")).filter(ImageFilter.MaxFilter(k))
    img.thumbnail((20, 20))                   # digit fits into 20x20 ...
    canvas = Image.new("L", (28, 28), 0)      # ... and sits in the centre of 28x28
    canvas.paste(img, ((28 - img.width) // 2, (28 - img.height) // 2))
    return (np.array(canvas, dtype=float) / 255.0).reshape(784, 1)


def one_hot(d):
    e = np.zeros((10, 1))
    e[d] = 1.0
    return e


def shifted(x, dx, dy):
    """Move a picture by dx, dy pixels (to get extra training copies)."""
    img = x.reshape(28, 28)
    out = np.zeros_like(img)
    ys, yd = (slice(0, 28 - dy), slice(dy, 28)) if dy >= 0 else (slice(-dy, 28), slice(0, 28 + dy))
    xs, xd = (slice(0, 28 - dx), slice(dx, 28)) if dx >= 0 else (slice(-dx, 28), slice(0, 28 + dx))
    out[yd, xd] = img[ys, xs]
    return out.reshape(784, 1)


def show_pictures(xs, titles):
    plt.figure(figsize=(1.6 * len(xs), 2))
    for i, (x, t) in enumerate(zip(xs, titles)):
        plt.subplot(1, len(xs), i + 1)
        plt.imshow(x.reshape(28, 28), cmap="gray")
        plt.title(t)
        plt.axis("off")
    plt.show()

## Step 1: Add your 10 pictures

Draw (or photograph) **one picture for each digit 0, 1, 2, ... 9**.

**Rules for good pictures**
- one digit per picture, thick line (marker or thick brush), digit fills most of the picture
- name the files exactly `0.png`, `1.png`, `2.png`, ... `9.png` (or `.jpg`)

**How to add them in Jupyter:** in the file list on the left, double-click the folder **`my_digits`**, then click the **Upload** button (arrow pointing up) and select all 10 files. No pictures yet? Change `DIGITS_FOLDER` below to `"sample_digits"` to use ready-made ones.

Then run the cell.

In [ ]:
DIGITS_FOLDER = "my_digits"          # use "sample_digits" for ready-made pictures
os.makedirs(DIGITS_FOLDER, exist_ok=True)

my_images = []                       # list of (picture, label)
for path in sorted(glob.glob(os.path.join(DIGITS_FOLDER, "*"))):
    name = os.path.basename(path).lower()
    if re.fullmatch(r"\d\.(png|jpg|jpeg)", name):
        my_images.append((prepare_image(path), int(name[0])))

if not my_images:
    raise RuntimeError("No pictures found in folder '" + DIGITS_FOLDER + "'. Upload 0.png ... 9.png there and run this cell again.")

print(len(my_images), "pictures ready.")
show_pictures([x for x, _ in my_images], [f"label {d}" for _, d in my_images])

**Check:** do the small pictures look like MNIST digits (thick, white, in the middle, on black)? If not, redraw them thicker and upload again.

## Step 2: Choose the network settings

Change the numbers below. Each one is a decision a real AI engineer makes.

| Setting | What it means | Try |
|---|---|---|
| `HIDDEN_LAYERS` | How many hidden layers, and how many neurons in each. `[30]` = one layer of 30 neurons, `[30, 15]` = two layers | `[10]`, `[30]`, `[100]`, `[30, 15]` |
| `LEARNING_RATE` | How big each learning step is | `0.5`, `3.0`, `10.0`, `100.0` |
| `EPOCHS` | How many times the network studies all the pictures | `1`, `5`, `20` |
| `BATCH_SIZE` | How many pictures it looks at before it corrects itself | `1`, `10`, `100` |
| `MNIST_SIZE` | How many MNIST pictures to learn from (max 50000) | `500`, `10000`, `50000` |
| `MY_COPIES` | How many slightly moved copies of each of *your* pictures to add | `0`, `30`, `200` |

In [ ]:
HIDDEN_LAYERS = [30]
LEARNING_RATE = 3.0
EPOCHS = 5
BATCH_SIZE = 10
MNIST_SIZE = 10000
MY_COPIES = 30

## Step 3: Train the network

The network starts with random weights and gradually corrects them. After every epoch it is tested on **10,000 MNIST pictures it has never seen**: `Epoch 2 : 9100 / 10000` means 9,100 digits were recognised correctly.

In [ ]:
if "my_images" not in globals() or "HIDDEN_LAYERS" not in globals():
    raise RuntimeError("Run Step 1 (upload pictures) and Step 2 (settings) first, then run this cell again.")

sizes = [784] + HIDDEN_LAYERS + [10]
print("Network layers:", sizes)

train = mnist_train[:MNIST_SIZE]
for x, d in my_images:
    for i in range(MY_COPIES + 1):
        dx, dy = (0, 0) if i == 0 else (random.randint(-2, 2), random.randint(-2, 2))
        train.append((shifted(x, dx, dy), one_hot(d)))
random.shuffle(train)
print("Training pictures:", len(train), f"({MNIST_SIZE} MNIST + {len(my_images) * (MY_COPIES + 1)} of yours)")

net = network.Network(sizes)
net.SGD(train, EPOCHS, BATCH_SIZE, LEARNING_RATE, test_data=mnist_test)

mnist_acc = net.evaluate(mnist_test) / len(mnist_test) * 100
my_ok = sum(int(np.argmax(net.feedforward(x))) == d for x, d in my_images)
print(f"\nMNIST test accuracy: {mnist_acc:.1f}%")
print(f"Your own 10 pictures recognised: {my_ok} / {len(my_images)}")

RESULTS.append({"layers": str(HIDDEN_LAYERS), "rate": LEARNING_RATE, "epochs": EPOCHS, "batch": BATCH_SIZE,
                "mnist": MNIST_SIZE, "copies": MY_COPIES, "mnist_acc": round(mnist_acc, 1),
                "my_ok": f"{my_ok}/{len(my_images)}", "new_image": "-"})

## Step 4: Test it on a NEW picture

Draw **one new digit** that the network has never seen (not one of your 10 training pictures). Upload it to the folder `my_digits` as `new.png` (for the ready-made pictures: `sample_digits/new.png`). Write the correct digit in `TRUE_DIGIT` and run the cell.

In [ ]:
TRUE_DIGIT = 2                          # <-- the digit you drew
NEW_IMAGE = DIGITS_FOLDER + "/new.png"  # <-- the file you uploaded

if not os.path.exists(NEW_IMAGE):
    raise RuntimeError("File " + NEW_IMAGE + " not found. Upload your new picture there first.")

x = prepare_image(NEW_IMAGE)
output = net.feedforward(x)
guess = int(np.argmax(output))

plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.imshow(x.reshape(28, 28), cmap="gray")
plt.title(f"The network sees this -> it says: {guess}")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.bar(range(10), output.flatten())
plt.xticks(range(10))
plt.title("How sure is it about each digit?")
plt.show()

verdict = "CORRECT" if guess == TRUE_DIGIT else "WRONG"
print(f"Network says {guess}, true digit is {TRUE_DIGIT}: {verdict}")
RESULTS[-1]["new_image"] = f"{guess} ({verdict})"

The bars show 10 numbers, one per digit. The tallest bar is the network's answer. If two bars are almost equal, the network is *unsure*.

## Step 5: Tune the network and compare

1. Go back to **Step 2** and change **one** setting only.
2. Run **Step 3** again (training), then **Step 4** again (new picture).
3. Run the cell below to see all your experiments in one table.

Change one thing at a time. Otherwise you will not know which change made the difference.

**Note:** the network starts with random weights, so even with the *same* settings two runs can give different results (sometimes quite different). Before you decide that a setting made a difference, run it twice.

In [ ]:
def show_results():
    head = ["#", "layers", "rate", "epochs", "batch", "MNIST", "copies", "MNIST acc %", "my 10", "new picture"]
    print("{:>2} {:<10} {:>6} {:>6} {:>5} {:>6} {:>6} {:>11} {:>6}  {}".format(*head))
    for i, r in enumerate(RESULTS, 1):
        print("{:>2} {:<10} {:>6} {:>6} {:>5} {:>6} {:>6} {:>11} {:>6}  {}".format(
            i, r["layers"], r["rate"], r["epochs"], r["batch"], r["mnist"], r["copies"],
            r["mnist_acc"], r["my_ok"], r["new_image"]))

show_results()

## Think about it

- Which setting changed the MNIST accuracy the most? Which changed almost nothing?
- Did the network recognise your *training* pictures better than your *new* picture? Why?
- What happens with `MY_COPIES = 0`? With `LEARNING_RATE = 100`? With `MNIST_SIZE = 500`?
- Would the network read a friend's handwriting as well as yours? How could you improve it?